# Qwen3-TTS on Google Colab

Run Qwen3-TTS with CUDA GPU acceleration on Google Colab.

**Requirements:** A Colab runtime with GPU (T4 or better).

**Setup:** Upload the project folder to Google Drive at `My Drive/Qwen3-TTS_UserFiles/`, then run the cells below.

## Setup

In [ ]:
import os

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Link project from Drive to expected location
PROJECT_DIR = '/content/drive/My Drive/Qwen3-TTS_UserFiles'
HOME_DIR = os.path.expanduser('~/Qwen3-TTS_UserFiles')

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project not found at '{PROJECT_DIR}'.\n"
        "Upload the Qwen3-TTS_UserFiles folder to 'My Drive/Qwen3-TTS_UserFiles/' in Google Drive."
    )

if not os.path.exists(HOME_DIR):
    os.symlink(PROJECT_DIR, HOME_DIR)
    print(f'Linked {PROJECT_DIR} -> {HOME_DIR}')

# Install system dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null

# Install Python dependencies
!pip install -q torch>=2.0 qwen-tts transformers==4.57.3 flask librosa soundfile numpy pydub requests gradio accelerate bitsandbytes scipy

print('Setup complete!')

In [ ]:
# Configure for CUDA backend
import json

config_path = os.path.expanduser('~/Qwen3-TTS_UserFiles/config.json')
with open(config_path) as f:
    config = json.load(f)

config['advanced']['backend'] = 'torch'
config['advanced']['dtype'] = 'float16'

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Start the TTS server in the background
import subprocess, time, requests, sys

sys.path.insert(0, os.path.expanduser('~/Qwen3-TTS_UserFiles'))

server = subprocess.Popen(
    ['python', os.path.expanduser('~/Qwen3-TTS_UserFiles/voice_server.py')],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for server to be ready (up to 60 seconds)
server_url = 'http://127.0.0.1:5123'
print('Starting server...', end='')
for i in range(60):
    # Check if process crashed
    if server.poll() is not None:
        print(f'\nServer exited with code {server.returncode}')
        print('STDERR:', server.stderr.read().decode()[-2000:])
        raise RuntimeError('Server failed to start — check errors above')
    try:
        resp = requests.get(f'{server_url}/health', timeout=1)
        if resp.status_code == 200:
            print(f'\nServer ready! (took {i+1}s)')
            health = resp.json()
            print(f'  Backend: {health.get("backend", "N/A")}')
            print(f'  Model size: {health.get("model_size", "N/A")}')
            break
    except requests.ConnectionError:
        pass
    print('.', end='', flush=True)
    time.sleep(1)
else:
    print('\nServer did not respond after 60s')
    print('STDERR:', server.stderr.read().decode()[-2000:])

In [ ]:
# Launch Gradio UI with public URL
import sys
sys.path.insert(0, os.path.expanduser('~/Qwen3-TTS_UserFiles'))

from voice_ui import build_ui
demo = build_ui()
demo.launch(server_name='0.0.0.0', share=True)

In [ ]:
# Quick generation example (without UI)
from voice_client import TTSClient

client = TTSClient()
output = client.generate(
    'Hello from Google Colab! This is Qwen3 TTS running on a GPU.',
    mode='design',
    description='A warm, friendly voice with clear articulation',
    output='colab_test.wav'
)
print(f'Generated: {output}')

# Play in notebook
from IPython.display import Audio
Audio(output)